<a href="https://colab.research.google.com/github/Jhoglund88/TriageFlow/blob/main/TriageFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Functions (Lesson 5)

In [1]:
import csv
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("GROQ_API_KEY")

def create_folders():
    os.makedirs("data", exist_ok=True)
    os.makedirs("models", exist_ok=True)
    os.makedirs("output", exist_ok=True)

def get_patient_age():
  MAX_AGE = 120

  while True:
          try:
              patient_age = int(input("Ange din ålder (0-120): "))
              if 0 <= patient_age <= MAX_AGE:
                  return patient_age
              else:
                  print(f"[FEL] Ogiltig ålder! Åldern måste vara mellan 0 och {int(MAX_AGE)} år.")
          except ValueError:
              print("[FEL] Du måste ange ålder i siffror (t.ex. 25).")


def get_symptom_category():
  while True:
        try:
            print("\nVälj symptomkategori (1-4):")
            print("1. Luftvägar\n2. Sår/Skador\n3. Infektion/Feber\n4. Övrigt")
            symptom_category = int(input("Ditt val: "))
            if symptom_category == 1:
              return "Luftvägar"
            elif symptom_category == 2:
              return "Sår/Skador"
            elif symptom_category == 3:
              return "Infektion/Feber"
            elif symptom_category == 4:
              return "Övrigt"
            else:
                print("[FEL] Du måste välja ett alternativ mellan 1 och 4.")
        except ValueError:
            print("[FEL] Ange ditt val med en siffra (1, 2, 3 eller 4).")

def get_symptom_description():
  while True:
        symptom_description = input("\nBeskriv dina symptom (minst 5, max 500 tecken): ").strip()
        if len(symptom_description) < 5:
            print("[FEL] Beskrivningen är för kort. Beskriv dina besvär mer noggrant.")
        elif len(symptom_description) > 500:
            print("[FEL] Beskrivningen är för lång. Försök att korta ner den till max 500 tecken.")
        else:
            return symptom_description

def get_pain_scale():
  MAX_PAIN = 10
  while True:
      try:
          pain_scale = int(input(f"\nAnge din smärtgrad på en skala 0-10 (där {int(MAX_PAIN)} är värsta tänkbara smärta): "))
          if 0 <= pain_scale <= MAX_PAIN:
              return pain_scale
          else:
              print(f"[FEL] Ogiltig smärtgrad! Du måste ange en siffra mellan 0 och {int(MAX_PAIN)}.")
      except ValueError:
          print("[FEL] Ange din smärtgrad med en siffra (0-10).")



def save_patients(patients):
    filepath = os.path.join("data", "patients.csv")
    with open(filepath, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "age",
            "symptom_category",
            "symptom_description",
            "pain_scale",
            "emergency_reason"
        ])

        for patient in patients:
            if patient.is_urgent():
                emergency_reason = patient.emergency_reason
            else:
                emergency_reason = ""

            writer.writerow([
                patient.age,
                patient.symptom_category,
                patient.symptom_description,
                patient.pain_scale,
                emergency_reason
            ])



## OOP - Classes and inheritance

In [2]:
class Patient:
  def __init__(self, age, symptom_category, symptom_description, pain_scale):
    self.age = age
    self.symptom_category = symptom_category
    self.symptom_description = symptom_description
    self.pain_scale = pain_scale

  def __str__(self):
    return f"Ålder: {self.age}, Symptom: {self.symptom_category}, Beskrivning: {self.symptom_description}, Smärta: {self.pain_scale}/10"

  def is_urgent(self):
    return self.pain_scale >= 8

class EmergencyPatient(Patient):
  def __init__(self, age ,symptom_category, symptom_description, pain_scale, emergency_reason):
    super().__init__(age, symptom_category, symptom_description, pain_scale)
    self.emergency_reason = emergency_reason

In [3]:
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def analyze_patient(patient):
    prompt = f"""Patientens ålder: {patient.age}
Symptomkategori: {patient.symptom_category}
Symptombeskrivning: {patient.symptom_description}
Smärtskala: {patient.pain_scale}/10

Bedöm patientens vårdbehov.

Svara med:
- urgency_level: ett heltal mellan 1 och 5
- recommendation: vad patienten bör göra
- reason: en kort motivering
"""
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    data = {
        "model": "openai/gpt-oss-120b",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    try:
        response = requests.post(
            GROQ_URL,
            headers=headers,
            json=data
        )

        response.raise_for_status()

    except requests.exceptions.RequestException as e:
        print(f"[FEL] Ett fel uppstod vid kommunikationen med API:et: {e}")
        return None

    result = response.json()
    ai_response = result["choices"][0]["message"]["content"]

    return ai_response


## Main Program

In [4]:
create_folders()

patients = []
emergency_patients = []

while True:
  add_new_patient = input("\nVill du lägga till en ny patient? (j/n): ").strip().lower()
  if add_new_patient == 'j':
    patient_age = get_patient_age()
    symptom_category = get_symptom_category()
    symptom_description = get_symptom_description()
    pain_scale = get_pain_scale()

    patient = Patient(patient_age, symptom_category, symptom_description, pain_scale)

    if patient.is_urgent():
      emergency_reason = input("Vad är ditt akuta symptom? ")
      patient = EmergencyPatient(
          patient_age,
          symptom_category,
          symptom_description,
          pain_scale,
          emergency_reason
      )

      emergency_patients.append(patient)

    patients.append(patient)

    print("\nPatient registrerad!")
  elif add_new_patient == 'n':

    break
  else:
    print("[FEL] Du måste ange 'j' eller 'n'.")

save_patients(patients)

for patient in patients:
    print(f"\nPatientens ålder: {patient.age}")
    print(f"Patientens symptomkategori: {patient.symptom_category}")
    print(f"Patientens symptombeskrivning: {patient.symptom_description}")
    print(f"Patientens smärtgrad: {patient.pain_scale}")

for patient in emergency_patients:
    print(f"\nAkut patientens ålder: {patient.age}")
    print(f"Akut patientens symptomkategori: {patient.symptom_category}")
    print(f"Akut patientens symptombeskrivning: {patient.symptom_description}")
    print(f"Akut patientens smärtgrad: {patient.pain_scale}")
    print(f"Akut patientens akuta symptom: {patient.emergency_reason}")


In [5]:
test_patient = Patient(
    35,
    "Luftvägar",
    "Har hosta och ont i halsen",
    4
)

ai_result = analyze_patient(test_patient)

print(ai_result)

**urgency_level:** 2  
**recommendation:** Vila, drick mycket vätska och använd receptfria smärtstillande/febernedsättande (t.ex. ibuprofen eller paracetamol) samt halstabletter. Gör ett snabbtest för COVID‑19/ influensa och håll koll på eventuellt feber, andningssvårigheter eller snabbt förvärrande symptom. Kontakta vårdcentralen inom de närmaste 24‑48 timmarna för vidare bedömning, särskilt om ont i halsen kvarstår längre än några dagar eller om du får hög feber eller svullnad i halsen.  
**reason:** Hostan och halsont med måttlig smärta (4/10) utan tydliga varningstecken (t.ex. andningssvårigheter, hög feber eller svullnad) bedöms som icke‑akut, men bör följas upp inom kort för att utesluta bakteriell faryngit eller viral infektion.
